# 02 - Storing vectors in Chroma

## Goal

Build two persistent collections (fixed and document-aware). Embeddings come from the OpenAI Embeddings API and are passed to Chroma explicitly, along with metadata we can filter on. The embedding model is read from `.env`, and the same one is used for queries.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
core_notebook_path = ROOT / "notebooks/_rag_demo_core.ipynb"
get_ipython().run_line_magic("run", f'"{core_notebook_path}"')
DATA = ROOT / "data"
ARTIFACTS = ROOT / "artifacts"
print(f"Project root: {ROOT}")
openai_settings = get_openai_settings(require_api_key=False)
print(f"OpenAI configured: {openai_settings.configured}; "
      f"embedding={openai_settings.embedding_model}; chat={openai_settings.chat_model}")


Project root: /Users/snomula/Downloads/sripal/Context Engineern-RAG
OpenAI configured: False; embedding=text-embedding-3-small; chat=gpt-5-mini


In [2]:
generate_pdf(DATA / "raw/northstar_safety_bulletin.pdf")

documents = load_corpus(DATA / "raw")
fixed = chunk_corpus(documents, "fixed", size=80, overlap=15)
aware = chunk_corpus(documents, "document_aware", max_words=110, overlap=15)
embedder = get_embedder()
print("Embedding model:", embedder.name)


Embedding model: hashing-384d


In [3]:
db_path = ARTIFACTS / "chroma"
fixed_store = ChromaVectorStore(db_path, collection_name("northstar-fixed-v2", embedder), embedder)
aware_store = ChromaVectorStore(db_path, collection_name("northstar-aware-v2", embedder), embedder)
fixed_store.upsert(fixed)
aware_store.upsert(aware)
print({"fixed_vectors": fixed_store.count(), "aware_vectors": aware_store.count(),
       "persisted_at": str(db_path)})


{'fixed_vectors': 24, 'aware_vectors': 43, 'persisted_at': '/Users/snomula/Downloads/sripal/Context Engineern-RAG/artifacts/chroma'}


### What Chroma stores

Each record has an ID, the chunk text, its embedding and flat metadata. `PersistentClient` writes everything under `artifacts/chroma`, so the later notebooks can reuse it.

In [4]:
preview = aware_store.collection.peek(limit=3)
for chunk_id, text, metadata in zip(preview["ids"], preview["documents"], preview["metadatas"]):
    print("\nID:", chunk_id)
    print("METADATA:", metadata)
    print("TEXT:", text[:220])



ID: incident-search-2026-01-22:aware:000:00
METADATA: {'doc_type': 'markdown', 'heading': 'Postmortem: Search API latency on 2026-01-22', 'element_index': 0, 'strategy': 'document_aware', 'doc_id': 'incident-search-2026-01-22', 'element_type': 'section', 'parent_id': 'incident-search-2026-01-22:parent:000', 'path': '/Users/snomula/Downloads/sripal/Context Engineern-RAG/data/raw/incident_postmortem.md', 'chunk_id': 'incident-search-2026-01-22:aware:000:00', 'source': 'incident_postmortem.md'}
TEXT: Section: Postmortem: Search API latency on 2026-01-22
Document ID: `incident-search-2026-01-22`  
Classification: Internal  
Severity: SEV-2

ID: incident-search-2026-01-22:aware:001:00
METADATA: {'heading': 'Summary', 'element_index': 1, 'doc_type': 'markdown', 'doc_id': 'incident-search-2026-01-22', 'path': '/Users/snomula/Downloads/sripal/Context Engineern-RAG/data/raw/incident_postmortem.md', 'source': 'incident_postmortem.md', 'chunk_id': 'incident-search-2026-01-22:aware:001:00', 'stra

In [5]:
question = "What should I inspect when an Orion shows E17?"
for result in aware_store.query(question, k=4):
    print(f"{result.rank}. score={result.score:.3f} doc={result.chunk.doc_id} "
          f"type={result.chunk.metadata.get('element_type')}")
    print("  ", result.chunk.text[:200])


1. score=0.219 doc=support-tickets-2026-02 type=json_record
   ticket_id: NS-1042 | product: Orion X2 | region: Europe | severity: high | symptoms: E17 after heavy rain; intermittent power reduction | diagnosis: Moisture found in motor temperature sensor connecto
2. score=0.177 doc=handbook-2026 type=section
   Section: Hybrid work
Employees assigned to the Seattle office may work remotely up to **three days per week**. Tuesday is the common collaboration day and requires in-office attendance unless a direct
3. score=0.174 doc=incident-search-2026-01-22 type=section
   Section: Corrective actions
| Action | Owner | Due date | Status |
| Add a load-test gate for index-mode changes | Search Platform | 2026-02-14 | Complete |
| Alert when exact search is enabled above 
4. score=0.155 doc=financials-2025 type=table_row
   quarter: 2025-Q4 | region: Europe | product: Lyra Battery | revenue_usd: 1650000 | cost_usd: 990000 | units: 5500 | source_status: provisional


### Metadata filtering

Filters make sense when the app already knows a hard constraint (file type, tenant, region, product, security label). Don't use them to guess constraints the user never gave.

In [6]:
results = aware_store.query("rear axle torque", k=3, where={"doc_type": "html"})
[(r.chunk.doc_id, r.chunk.metadata.get("heading"), round(r.score, 3)) for r in results]


[('manual-orion-x2', 'Torque specifications', 0.358),
 ('manual-orion-x2', 'Firmware update', 0.216),
 ('manual-orion-x2', 'Torque specifications', 0.2)]

## Note on production

`PersistentClient` is fine for a local demo. For a multi-user setup you'd run Chroma as a server and plan for auth, tenancy, backups, monitoring and re-indexing when models change.